This notebook follows JAX 101: expressing computations, which can be find at https://docs.jax.dev/en/latest/101/arrays.html.

# 1. Arrays and jax.numpy

## From NumPy to JAX

In [1]:
import jax
import jax.numpy as jnp
import numpy as np

With this import, array creation, arithmetic, indexing, reductions, and so on all look just like NumPy:

In [2]:
x = jnp.arange(9.0).reshape(3, 3)
y = jnp.ones(3)

print(x @ y)
print(x.sum(axis=0))
print(x[1, :2])

[ 3. 12. 21.]
[ 9. 12. 15.]
[3. 4.]


The arrays themselves are instances of jax.Array:

In [3]:
isinstance(x, jax.Array)

True

Every JAX value also has a JAX type, which you can inspect with jax.typeof(). For an array, the JAX type roughly means its shape and dtype:

In [4]:
print(jax.typeof(x))

float32[3,3]


JAX arrays and NumPy arrays can often be used interchangeably

## JAX arrays are immutable

The most important difference from NumPy: once created, a JAX array’s contents cannot be changed. NumPy allows in-place mutation:

In [5]:
x_np = np.arange(10)
x_np[0] = 10
print(x_np)

[10  1  2  3  4  5  6  7  8  9]


The equivalent raises an error in JAX:

In [7]:
# x = jnp.arange(10)
# x[0] = 10

Instead, JAX provides functional updates via the at property, which return a new array with the update applied:

In [8]:
y = x.at[0].set(10)
print(x)  # unchanged
print(y)

[0 1 2 3 4 5 6 7 8 9]
[10  1  2  3  4  5  6  7  8  9]


Alongside set, there’s a family of update operations, including add, multiply, min, and max:

In [9]:
print(x.at[3].add(100))
print(x.at[:3].max(5))

[  0   1   2 103   4   5   6   7   8   9]
[5 5 5 3 4 5 6 7 8 9]


For when mutable arrays are really necessary, JAX provides a distinct array reference type, covered in https://docs.jax.dev/en/latest/101/state.html#jax-101-refs.

## Default dtypes and precision

NumPy defaults to 64-bit floating point. JAX defaults to 32-bit, which is usually what you want on accelerators like GPUs and TPUs:



In [10]:
print(np.array([1.0, 2.0]).dtype)
print(jnp.array([1.0, 2.0]).dtype)

float64
float32


To enable 64-bit floating points:

In [11]:
# jax.config.update("jax_enable_x64", True)

When operations mix dtypes, JAX applies type promotion rules that are similar to NumPy’s but not identical. In particular they’re designed to avoid accidentally promoting everything to 64-bit

## Indexing differences

JAX supports NumPy-style indexing, including slices and advanced integer array indexing. But some edge-case behaviors differ, because JAX is designed so that every operation can be compiled for accelerators, where raising an exception from inside a computation isn’t an option.

Most notably, out-of-bounds indexing doesn’t raise an error. When reading, the index is clamped to the bounds of the array: 

In [18]:
x = jnp.arange(10)
print(x.shape)
print(x[12])  # clamped to x[9]

(10,)
9


When writing with .at, out-of-bounds updates are dropped:

In [19]:
x.at[11].set(99)  # update is dropped

Array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32)

Another small difference: jax.numpy functions require arrays (or values of Python’s built-in numeric types) as inputs, rather than silently converting Python lists:

In [21]:
# jnp.sum([1, 2, 3])

This is deliberate. Silently converting lists to arrays is a common source of hidden performance problems, so JAX asks you to do it explicitly:

In [22]:
jnp.sum(jnp.array([1, 2, 3]))

Array(6, dtype=int32)

## Arrays live on device

A JAX array’s data lives on one or more devices, like CPU, GPU, or TPU. The same JAX code runs on all of them; JAX places arrays on the default device (typically your accelerator, if you have one) and computations follow their data. An array can even be sharded across many devices, so that JAX programs can be written once and run on one chip or thousands. Every array carries a sharding attribute describing exactly how its data is placed:

In [23]:
x.sharding

SingleDeviceSharding(device=CpuDevice(id=0), memory_kind=device)

# 2. Transformations: grad and vmap

## Automatic differentiation with jax.grad

jax.grad() takes a scalar-valued function and returns a new function that computes its gradient:

In [26]:
grad_tanh = jax.grad(jnp.tanh)
print(grad_tanh(2.0))

0.070650816


If f is a Python function that evaluates the mathematical function $f$, then jax.grad(f) is a Python function that evaluates $\nabla f$, so grad(f)(x) is the gradient value $\nabla f(x)$.

Since jax.grad maps functions to functions, you can apply it repeatedly to take higher-order derivatives:

In [27]:
f = lambda x: x**3 + 2*x**2 - 3*x + 1

dfdx = jax.grad(f)
d2fdx = jax.grad(dfdx)
d3fdx = jax.grad(d2fdx)

print(dfdx(1.0))   # 3x² + 4x - 3  evaluated at 1, i.e. 4
print(d2fdx(1.0))  # 6x + 4        evaluated at 1, i.e. 10
print(d3fdx(1.0))  # 6

4.0
10.0
6.0


## Differentiating with respect to different arguments

By default jax.grad differentiates with respect to the first argument. The argnums parameter selects other arguments, or several at once. Here’s a linear logistic regression model, where we might want gradients with respect to the weights W, the bias b, or both:

In [28]:
def sigmoid(x):
  return 0.5 * (jnp.tanh(x / 2) + 1)

# Outputs probability of a label being true.
def predict(W, b, inputs):
  return sigmoid(jnp.dot(inputs, W) + b)

# A toy dataset, and some parameter values.
inputs = jnp.array([[0.52, 1.12,  0.77],
                    [0.88, -1.08, 0.15],
                    [0.52, 0.06, -1.30],
                    [0.74, -2.49, 1.39]])
targets = jnp.array([True, True, False, True])
W = jnp.array([0.1, 0.4, -0.3])
b = 0.5

# Training loss is the negative log-likelihood of the training examples.
def loss(W, b):
  preds = predict(W, b, inputs)
  label_probs = preds * targets + (1 - preds) * (1 - targets)
  return -jnp.sum(jnp.log(label_probs))

In [29]:
# Differentiate `loss` with respect to the first positional argument:
W_grad = jax.grad(loss, argnums=0)(W, b)
print(f'{W_grad=}')

# Since argnums=0 is the default, this does the same thing:
W_grad = jax.grad(loss)(W, b)
print(f'{W_grad=}')

# But you can choose different values too, and drop the keyword:
b_grad = jax.grad(loss, 1)(W, b)
print(f'{b_grad=}')

# Including tuple values:
W_grad, b_grad = jax.grad(loss, (0, 1))(W, b)
print(f'{W_grad=}')
print(f'{b_grad=}')

W_grad=Array([-0.7203479,  1.937475 , -2.226889 ], dtype=float32)
W_grad=Array([-0.7203479,  1.937475 , -2.226889 ], dtype=float32)
b_grad=Array(-0.76290697, dtype=float32, weak_type=True)
W_grad=Array([-0.7203479,  1.937475 , -2.226889 ], dtype=float32)
b_grad=Array(-0.76290697, dtype=float32, weak_type=True)


Real models don’t keep their parameters in separate positional arguments. Instead they use nested containers like dictionaries. jax.grad handles those natively; that’s the subject of the next page, 

## value_and_grad and auxiliary outputs

You usually want the loss value as well as its gradient, for example to log training progress. jax.value_and_grad() computes both in one pass:

In [30]:
loss_value, Wb_grad = jax.value_and_grad(loss, (0, 1))(W, b)
print(loss_value)

3.5066924


And sometimes a function naturally computes intermediate results worth returning alongside the scalar being differentiated. Provide a function that produces a (scalar_output, aux_data) pair as output and pass has_aux=True:

In [31]:
def loss_and_preds(W, b):
  preds = predict(W, b, inputs)
  label_probs = preds * targets + (1 - preds) * (1 - targets)
  return -jnp.sum(jnp.log(label_probs)), preds

W_grad, preds = jax.grad(loss_and_preds, has_aux=True)(W, b)
print(preds)

[0.68330455 0.5277215  0.7243215  0.30174544]


## Checking derivatives numerically

In [32]:
eps = 1e-4
b_grad_numerical = (loss(W, b + eps / 2.) - loss(W, b - eps / 2.)) / eps
print('b_grad_numerical', b_grad_numerical)
print('b_grad_autodiff', jax.grad(loss, 1)(W, b))

b_grad_numerical -0.76293945
b_grad_autodiff -0.76290697


## Automatic vectorization with jax.vmap

jax.vmap() transforms a function written for single inputs into one that works efficiently on batches of inputs. Consider a function that computes the convolution of two one-dimensional vectors:

In [39]:
x = jnp.arange(5.0)
w = jnp.array([2., 3., 4.])

def convolve(x, w):
  output = []
  for i in range(1, len(x) - 1):
    output.append(jnp.dot(x[i-1:i+2], w))
  return jnp.array(output)

convolve(x, w)

Array([11., 20., 29.], dtype=float32)

Suppose we want to apply this function to a whole batch of xs and ws:

In [40]:
xs = jnp.stack([x, x])
ws = jnp.stack([w, w])

In [41]:
def manually_batched_convolve(xs, ws):
  output = []
  for i in range(xs.shape[0]):
    output.append(convolve(xs[i], ws[i]))
  return jnp.stack(output)

manually_batched_convolve(xs, ws)

Array([[11., 20., 29.],
       [11., 20., 29.]], dtype=float32)

jax.vmap does this rewrite automatically:

In [46]:
auto_batch_convolve = jax.vmap(convolve, (0,None))

auto_batch_convolve(xs, 2.)


Array([[[0., 2., 4.],
        [2., 4., 6.],
        [4., 6., 8.]],

       [[0., 2., 4.],
        [2., 4., 6.],
        [4., 6., 8.]]], dtype=float32)